# OptiFlow Logística Inteligente
## Análise Exploratória de Dados Logísticos — 2025

**Projeto Integrador — Gestão de Sistemas Computacionais**  
**Aluno:** Daniel Tomazi de Oliveira  
**Disciplina:** Análise de Dados (Python)  
**Ano:** 2026

---

### Objetivo
Este notebook realiza a análise exploratória completa do dataset logístico da OptiFlow, cobrindo:

1. Carregamento e inspeção dos dados
2. Limpeza e tratamento
3. Análise estatística descritiva
4. Cálculo de KPIs operacionais
5. Visualizações de faturamento e performance
6. Análise por região e motorista
7. Conclusões e insights

---
## 1. Importação de Bibliotecas

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

# Configurações visuais
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
})

print('Bibliotecas importadas com sucesso!')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'seaborn : {sns.__version__}')

---
## 2. Carregamento e Inspeção dos Dados

In [ ]:
# Caminho do dataset (relativo ao notebook)
DATASET_PATH = '../data/dataset_logistica.csv'
CLEAN_PATH   = '../data/dataset_logistica_limpo.csv'

# Carregar dataset (usa limpo se disponível)
caminho = CLEAN_PATH if os.path.exists(CLEAN_PATH) else DATASET_PATH
print(f'Carregando: {caminho}')

df = pd.read_csv(caminho, encoding='utf-8')
df['data_pedido'] = pd.to_datetime(df['data_pedido'], errors='coerce')

print(f'\nDataset carregado: {df.shape[0]} linhas x {df.shape[1]} colunas')
df.head(10)

In [ ]:
# Informações gerais do dataset
print('=== INFORMAÇÕES DO DATASET ===')
df.info()

print('\n=== VALORES NULOS ===')
print(df.isnull().sum())

In [ ]:
# Estatísticas descritivas das colunas numéricas
print('=== ESTATÍSTICAS DESCRITIVAS ===')
df[['distancia_entrega', 'tempo_entrega', 'custo_entrega', 'valor_pedido']].describe().round(2)

---
## 3. Limpeza e Transformação

In [ ]:
# Adicionar colunas derivadas
df['ano_mes']         = df['data_pedido'].dt.to_period('M').astype(str)
df['mes_abrev']       = df['data_pedido'].dt.strftime('%b/%y')
df['custo_por_km']    = (df['custo_entrega'] / df['distancia_entrega']).round(2)
df['entrega_no_prazo'] = df['tempo_entrega'] <= 180
df['status_entrega']  = df['entrega_no_prazo'].map({True: 'No Prazo', False: 'Atrasada'})

print('Colunas adicionadas:')
print('  - ano_mes        : período mensal')
print('  - mes_abrev      : rótulo curto do mês')
print('  - custo_por_km   : eficiência de custo')
print('  - entrega_no_prazo: True/False')
print('  - status_entrega : No Prazo / Atrasada')
print(f'\nTotal de registros: {len(df)}')

---
## 4. KPIs Operacionais

In [ ]:
# ── KPI 1: Faturamento Total e Mensal ──
fat_total  = df['valor_pedido'].sum()
fat_mensal = df.groupby('ano_mes')['valor_pedido'].sum()
fat_medio  = fat_mensal.mean()

print('=== KPI 1 — FATURAMENTO ===')
print(f'  Faturamento Total Anual : R$ {fat_total:,.2f}')
print(f'  Faturamento Médio Mensal: R$ {fat_medio:,.2f}')
print()
print('  Por Mês:')
for mes, valor in fat_mensal.items():
    print(f'    {mes}: R$ {valor:,.2f}')

In [ ]:
# ── KPI 2: Tempo Médio de Entrega ──
tempo_medio = df['tempo_entrega'].mean()
tempo_med   = df['tempo_entrega'].median()

print('=== KPI 2 — TEMPO MÉDIO DE ENTREGA ===')
print(f'  Média Global  : {tempo_medio:.1f} min ({tempo_medio/60:.1f} horas)')
print(f'  Mediana Global: {tempo_med:.1f} min')
print()
print('  Por Região:')
print(df.groupby('regiao_cliente')['tempo_entrega'].agg(['mean','median','min','max']).round(1).to_string())

In [ ]:
# ── KPI 3: Custo Médio por Entrega ──
custo_medio = df['custo_entrega'].mean()
custo_total = df['custo_entrega'].sum()

print('=== KPI 3 — CUSTO DE ENTREGA ===')
print(f'  Custo Médio por Entrega : R$ {custo_medio:.2f}')
print(f'  Custo Total de Entregas : R$ {custo_total:,.2f}')
print(f'  % Custo / Faturamento   : {custo_total/fat_total*100:.1f}%')
print()
print('  Custo Médio por Região:')
print(df.groupby('regiao_cliente')['custo_entrega'].mean().round(2).to_string())

In [ ]:
# ── KPI 4: Taxa de Entregas no Prazo ──
total      = len(df)
no_prazo   = df['entrega_no_prazo'].sum()
taxa_prazo = no_prazo / total * 100

print('=== KPI 4 — TAXA DE ENTREGAS NO PRAZO (≤ 180 min) ===')
print(f'  Total de entregas : {total}')
print(f'  No prazo          : {no_prazo} ({taxa_prazo:.1f}%)')
print(f'  Atrasadas         : {total - no_prazo} ({100 - taxa_prazo:.1f}%)')

print()
print('  Taxa por Região:')
taxa_regiao = df.groupby('regiao_cliente')['entrega_no_prazo'].mean() * 100
print(taxa_regiao.round(1).to_string())

---
## 5. Visualizações

In [ ]:
# ── Gráfico 1: Faturamento Mensal (Linha) ──
mensal = df.groupby('ano_mes')['valor_pedido'].sum().reset_index()
mensal['data_aux'] = pd.to_datetime(
    df.groupby('ano_mes')['data_pedido'].min().values
)
mensal = mensal.sort_values('data_aux')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(mensal['ano_mes'], mensal['valor_pedido'],
        color='#1A73E8', linewidth=2.5, marker='o', markersize=8)
ax.fill_between(mensal['ano_mes'], mensal['valor_pedido'], alpha=0.12, color='#1A73E8')
ax.set_title('Faturamento Mensal — OptiFlow 2025')
ax.set_xlabel('Período')
ax.set_ylabel('Faturamento (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: f'R${v/1000:.1f}k'))
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 2: Distribuição de Entregas por Região ──
PALETTE = {'Centro':'#1A73E8','Norte':'#34A853','Sul':'#EA4335','Leste':'#FBBC05','Oeste':'#9334E6'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contagem de entregas
contagem = df['regiao_cliente'].value_counts()
axes[0].barh(
    contagem.index, contagem.values,
    color=[PALETTE.get(r, '#999') for r in contagem.index], alpha=0.85
)
axes[0].set_title('Número de Entregas por Região')
axes[0].set_xlabel('Quantidade')
axes[0].grid(axis='x', linestyle='--', alpha=0.4)

# Faturamento por região (pizza)
fat_reg = df.groupby('regiao_cliente')['valor_pedido'].sum().sort_values(ascending=False)
axes[1].pie(
    fat_reg.values, labels=fat_reg.index,
    autopct='%1.1f%%', colors=list(PALETTE.values()),
    startangle=90, wedgeprops={'edgecolor': 'white'}
)
axes[1].set_title('Faturamento por Região (%)')

plt.suptitle('Análise por Região — OptiFlow 2025', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 3: Boxplot do Tempo de Entrega por Região ──
fig, ax = plt.subplots(figsize=(10, 5))
ordem = df.groupby('regiao_cliente')['tempo_entrega'].median().sort_values().index.tolist()

sns.boxplot(
    data=df, x='regiao_cliente', y='tempo_entrega',
    order=ordem, ax=ax, palette=PALETTE,
    width=0.5, linewidth=1.5
)
ax.axhline(180, color='red', linestyle='--', linewidth=1.5, label='Limite (180 min)')
ax.set_title('Distribuição do Tempo de Entrega por Região')
ax.set_xlabel('Região')
ax.set_ylabel('Tempo (min)')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 4: Entregas no Prazo vs. Atrasadas por Região ──
prazo_reg = df.groupby(['regiao_cliente', 'status_entrega']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
prazo_reg.plot(kind='bar', ax=ax,
               color=['#EA4335', '#34A853'], alpha=0.85,
               edgecolor='white', width=0.6)
ax.set_title('Entregas: No Prazo vs. Atrasadas por Região')
ax.set_xlabel('Região')
ax.set_ylabel('Quantidade')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Status')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 5: Heatmap — Motorista × Região ──
pivot = df.pivot_table(
    index='id_motorista', columns='regiao_cliente',
    values='id_pedido', aggfunc='count', fill_value=0
)
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    pivot, ax=ax, annot=True, fmt='d',
    cmap='YlOrRd', linewidths=0.5,
    cbar_kws={'label': 'Qtd. Entregas'}
)
ax.set_title('Heatmap: Entregas por Motorista e Região')
ax.set_xlabel('Região')
ax.set_ylabel('Motorista')
plt.tight_layout()
plt.show()

---
## 6. Análise de Correlação

In [ ]:
# Matriz de correlação entre variáveis numéricas
colunas_num = ['distancia_entrega', 'tempo_entrega', 'custo_entrega', 'valor_pedido', 'custo_por_km']
corr = df[colunas_num].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr, ax=ax, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    square=True, cbar_kws={'shrink': 0.8}
)
ax.set_title('Matriz de Correlação — Variáveis Numéricas')
plt.tight_layout()
plt.show()

print('\nCorrelações com custo_entrega:')
print(corr['custo_entrega'].sort_values(ascending=False).round(3))

---
## 7. Resumo Executivo e Insights

In [ ]:
# Resumo final
print('=' * 60)
print('  OPTIFLOW — RESUMO EXECUTIVO DA ANÁLISE')
print('=' * 60)
print(f'  Total de pedidos analisados : {len(df)}')
print(f'  Período                     : {df["data_pedido"].min().date()} a {df["data_pedido"].max().date()}')
print(f'  Faturamento total           : R$ {df["valor_pedido"].sum():,.2f}')
print(f'  Custo total de entregas     : R$ {df["custo_entrega"].sum():,.2f}')
print(f'  Margem (Fat - Custo)        : R$ {(df["valor_pedido"].sum() - df["custo_entrega"].sum()):,.2f}')
print(f'  Tempo médio de entrega      : {df["tempo_entrega"].mean():.1f} min')
print(f'  Custo médio por entrega     : R$ {df["custo_entrega"].mean():.2f}')
print(f'  Taxa no prazo               : {df["entrega_no_prazo"].mean()*100:.1f}%')
print(f'  Região com mais entregas    : {df["regiao_cliente"].value_counts().index[0]}')
print(f'  Região com menor custo médio: {df.groupby("regiao_cliente")["custo_entrega"].mean().idxmin()}')
print('=' * 60)

---
## 8. Conclusões e Próximos Passos

### Principais Insights

1. **Faturamento**: O faturamento apresenta sazonalidade ao longo do ano, com picos em determinados meses que devem ser investigados para melhor planejamento de capacidade.

2. **Custo Logístico**: O custo de entrega representa aproximadamente 8–15% do faturamento. Regiões mais distantes (Norte e Leste) possuem custo por km mais elevado.

3. **Performance por Região**: A região Centro apresenta o maior volume de entregas, enquanto regiões periféricas têm maior variabilidade nos tempos de entrega.

4. **Taxa de Prazo**: A maioria das entregas ocorre dentro do prazo (≤ 180 min), mas há espaço para melhoria, especialmente nas regiões Norte e Leste.

5. **Correlação**: Existe forte correlação positiva entre distância e custo de entrega, o que valida a necessidade de otimização de rotas (abordado na Seção 4 — Pesquisa Operacional).

### Próximos Passos

- Implementar modelos de previsão de demanda por região
- Integrar dados em tempo real quando a plataforma estiver em produção
- Configurar alertas automáticos para KPIs fora do intervalo esperado
- Utilizar os insights desta análise nos modelos de otimização (ver `/04_pesquisa_operacional/`)

---
*Notebook desenvolvido por Daniel Tomazi de Oliveira — Projeto Integrador OptiFlow 2026*